In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import rankdata
import pymannkendall as mk


# =========================================================
# 1. 基础函数：Pettitt 检验
# =========================================================
def pettitt_test(x):
    """
    Pettitt change-point test

    Parameters
    ----------
    x : array-like
        时间序列数值

    Returns
    -------
    dict
        {
            'K': 统计量,
            'cp_index': 突变点索引(0-based),
            'p_value': p值
        }
    """
    x = np.asarray(x, dtype=float)
    n = len(x)

    if n < 2:
        return {'K': np.nan, 'cp_index': np.nan, 'p_value': np.nan}

    r = rankdata(x)
    U = np.zeros(n)

    for t in range(n):
        U[t] = 2 * np.sum(r[:t + 1]) - (t + 1) * (n + 1)

    K = np.max(np.abs(U))
    cp_index = int(np.argmax(np.abs(U)))
    p_value = 2 * np.exp((-6 * K ** 2) / (n ** 3 + n ** 2))

    return {
        'K': K,
        'cp_index': cp_index,
        'p_value': p_value
    }


# =========================================================
# 2. 基础函数：顺序 Mann-Kendall 突变检验
# =========================================================
def sequential_mk_test(x):
    """
    计算顺序 Mann-Kendall 检验的 UF / UB 序列

    Parameters
    ----------
    x : array-like
        时间序列数值

    Returns
    -------
    uf : np.ndarray
    ub : np.ndarray
    """
    x = np.asarray(x, dtype=float)
    n = len(x)

    uf = np.zeros(n)
    s_k = np.zeros(n)

    for i in range(1, n):
        count = 0
        for j in range(i):
            if x[i] > x[j]:
                count += 1
        s_k[i] = s_k[i - 1] + count
        E = i * (i + 1) / 4
        Var = i * (i + 1) * (2 * i + 5) / 72
        uf[i] = 0 if Var == 0 else (s_k[i] - E) / np.sqrt(Var)

    x_rev = x[::-1]
    ub_rev = np.zeros(n)
    s_k2 = np.zeros(n)

    for i in range(1, n):
        count = 0
        for j in range(i):
            if x_rev[i] > x_rev[j]:
                count += 1
        s_k2[i] = s_k2[i - 1] + count
        E = i * (i + 1) / 4
        Var = i * (i + 1) * (2 * i + 5) / 72
        ub_rev[i] = 0 if Var == 0 else (s_k2[i] - E) / np.sqrt(Var)

    ub = -ub_rev[::-1]
    return uf, ub


# =========================================================
# 3. 数据预处理函数
# =========================================================
def prepare_data(
    file_path,
    sep=",",
    col_map=None,
    year_max=None,
    dropna=True
):
    """
    读取并标准化数据

    Parameters
    ----------
    file_path : str
        输入文件路径
    sep : str
        分隔符，csv一般为','，制表符文件为'\\t'
    col_map : dict
        列名映射，例如：
        {
            "group": "pfas",
            "time": "year",
            "value": "mean_mean"
        }
    year_max : int or None
        若不为 None，则保留 time <= year_max
    dropna : bool
        是否删除关键列缺失行

    Returns
    -------
    pd.DataFrame
        标准化后的数据，列名统一为:
        ['group', 'time', 'value']
    """
    if col_map is None:
        col_map = {
            "group": "pfas",
            "time": "year",
            "value": "mean_mean"
        }

    df = pd.read_csv(file_path, sep=sep)
    df.columns = df.columns.str.strip()

    required_cols = [col_map["group"], col_map["time"], col_map["value"]]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"缺少必要列: {missing_cols}，当前列为: {df.columns.tolist()}")

    df = df[required_cols].copy()
    df = df.rename(columns={
        col_map["group"]: "group",
        col_map["time"]: "time",
        col_map["value"]: "value"
    })

    df["time"] = pd.to_numeric(df["time"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    if year_max is not None:
        df = df[df["time"] <= year_max].copy()
    df = df[df["time"] > 2002].copy()
    if dropna:
        df = df.dropna(subset=["group", "time", "value"])

    df = df.sort_values(["group", "time"]).reset_index(drop=True)
    return df


# =========================================================
# 4. 单组分析函数
# =========================================================
def analyze_one_group(group_df, group_name):
    """
    对单个分组执行：
    1) MK 趋势检验
    2) 顺序 MK 突变检验
    3) Pettitt 检验

    Parameters
    ----------
    group_df : pd.DataFrame
        必须包含 ['group', 'time', 'value']
    group_name : str
        当前组名称

    Returns
    -------
    trend_result : dict
    change_result : dict
    seqmk_df : pd.DataFrame
    """
    group_df = group_df.sort_values("time").copy()
    times = group_df["time"].to_numpy()
    values = group_df["value"].to_numpy()

    if len(values) < 5:
        return None, None, None

    # MK 趋势检验
    mk_result = mk.original_test(values)
    trend_result = {
        "group": group_name,
        "n": len(values),
        "mk_trend": mk_result.trend,
        "mk_h": mk_result.h,
        "mk_p": mk_result.p,
        "mk_z": mk_result.z,
        "mk_tau": mk_result.Tau,
        "mk_s": mk_result.s,
        "mk_var_s": mk_result.var_s,
        "sen_slope": mk_result.slope,
        "sen_intercept": mk_result.intercept
    }

    # 顺序 MK
    uf, ub = sequential_mk_test(values)
    seqmk_df = pd.DataFrame({
        "group": group_name,
        "time": times,
        "value": values,
        "UF": uf,
        "UB": ub
    })

    diff = np.abs(uf - ub)
    seq_cp_index = int(np.argmin(diff))
    seq_cp_time = times[seq_cp_index]

    # Pettitt
    pettitt_res = pettitt_test(values)
    cp_index = pettitt_res["cp_index"]
    cp_time = times[cp_index] if pd.notna(cp_index) else np.nan

    change_result = {
        "group": group_name,
        "n": len(values),
        "pettitt_K": pettitt_res["K"],
        "pettitt_p": pettitt_res["p_value"],
        "pettitt_cp_index": cp_index,
        "pettitt_cp_time": cp_time,
        "seqMK_cp_index": seq_cp_index,
        "seqMK_cp_time": seq_cp_time
    }

    return trend_result, change_result, seqmk_df


# =========================================================
# 5. 主分析函数
# =========================================================
def run_change_point_analysis(
    file_path,
    output_dir=None,
    sep=",",
    col_map=None,
    year_max=None,
    min_n=5,
    output_prefix="change_point_result"
):
    """
    读取数据后，按 group 分组执行：
    - Mann-Kendall 趋势检验
    - 顺序 Mann-Kendall 突变检验
    - Pettitt 检验

    Parameters
    ----------
    file_path : str
        输入文件
    output_dir : str or None
        输出目录；若为 None，则默认输出到输入文件所在目录
    sep : str
        分隔符
    col_map : dict
        列名映射，如：
        {
            "group": "pfas",
            "time": "year",
            "value": "mean_mean"
        }
    year_max : int or None
        时间筛选上限，例如 2020 表示只保留 <= 2020
    min_n : int
        每组最少样本数
    output_prefix : str
        输出文件名前缀

    Returns
    -------
    dict
        {
            "data": 标准化数据,
            "trend": 趋势结果表,
            "change": 突变结果表,
            "seqmk": 顺序MK结果表,
            "excel_path": 输出Excel路径
        }
    """
    df = prepare_data(
        file_path=file_path,
        sep=sep,
        col_map=col_map,
        year_max=year_max,
        dropna=True
    )

    trend_results = []
    change_results = []
    seqmk_list = []

    for group_name, g in df.groupby("group"):
        if len(g) < min_n:
            print(f"[跳过] group={group_name}，样本数={len(g)} < min_n={min_n}")
            continue

        trend_result, change_result, seqmk_df = analyze_one_group(g, group_name)

        if trend_result is not None:
            trend_results.append(trend_result)
        if change_result is not None:
            change_results.append(change_result)
        if seqmk_df is not None:
            seqmk_list.append(seqmk_df)

    trend_df = pd.DataFrame(trend_results)
    change_df = pd.DataFrame(change_results)
    seqmk_df = pd.concat(seqmk_list, ignore_index=True) if seqmk_list else pd.DataFrame()

    if output_dir is None:
        output_dir = os.path.dirname(file_path)

    os.makedirs(output_dir, exist_ok=True)
    excel_path = os.path.join(output_dir, f"{output_prefix}.xlsx")

    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="filtered_data", index=False)
        trend_df.to_excel(writer, sheet_name="MK_trend", index=False)
        change_df.to_excel(writer, sheet_name="ChangePoint", index=False)
        seqmk_df.to_excel(writer, sheet_name="SeqMK_UF_UB", index=False)

    print(f"分析完成，结果已保存到: {excel_path}")

    return {
        "data": df,
        "trend": trend_df,
        "change": change_df,
        "seqmk": seqmk_df,
        "excel_path": excel_path
    }

In [6]:

result = run_change_point_analysis(
    file_path=r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_short_g.csv",
    output_dir=r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\stats",
    sep=",",
    col_map={
        "group": "pfas",
        "time": "year",
        "value": "mean_mean"
    },
    year_max=2020,
    min_n=5,
    output_prefix="pfas_mk_pettitt_g"
)

print(result["trend"])
print(result["change"])

分析完成，结果已保存到: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\stats\pfas_mk_pettitt_g.xlsx
      group   n    mk_trend  mk_h      mk_p      mk_z    mk_tau  mk_s  \
0  lc_value  18  increasing  True  0.005064  2.802950  0.490196  75.0   
1  sc_value  18  increasing  True  0.000277  3.636259  0.633987  97.0   
2     value  18  increasing  True  0.001124  3.257482  0.568627  87.0   

   mk_var_s  sen_slope  sen_intercept  
0     697.0   0.146766       1.739260  
1     697.0   0.371221       0.327648  
2     697.0   0.552737       1.771529  
      group   n  pettitt_K  pettitt_p  pettitt_cp_index  pettitt_cp_time  \
0  lc_value  18       80.0   0.003908                 9             2012   
1  sc_value  18       81.0   0.003341                 8             2011   
2     value  18       81.0   0.003341                 8             2011   

   seqMK_cp_index  seqMK_cp_time  
0              10           2013  
1               9           2012  
2              14           2017  


In [2]:

result = run_change_point_analysis(
    file_path=r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_short_developing.csv",
    output_dir=r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\stats",
    sep=",",
    col_map={
        "group": "pfas",
        "time": "year",
        "value": "mean_mean"
    },
    year_max=2020,
    min_n=5,
    output_prefix="pfas_mk_pettitt_developing"
)

print(result["trend"])
print(result["change"])

分析完成，结果已保存到: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\stats\pfas_mk_pettitt_developing.xlsx
      group   n    mk_trend  mk_h          mk_p      mk_z    mk_tau   mk_s  \
0  lc_value  18  increasing  True  1.520054e-04  3.787770  0.660131  101.0   
1  sc_value  18  increasing  True  8.474900e-07  4.924101  0.856209  131.0   
2     value  18  increasing  True  7.837748e-06  4.469569  0.777778  119.0   

   mk_var_s  sen_slope  sen_intercept  
0     697.0   0.449929       2.746730  
1     697.0   0.851526       1.697119  
2     697.0   1.307197       4.349605  
      group   n  pettitt_K  pettitt_p  pettitt_cp_index  pettitt_cp_time  \
0  lc_value  18       81.0   0.003341                 8             2011   
1  sc_value  18       81.0   0.003341                 8             2011   
2     value  18       81.0   0.003341                 8             2011   

   seqMK_cp_index  seqMK_cp_time  
0               8           2011  
1              10           2013  
2             

In [3]:

result = run_change_point_analysis(
    file_path=r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_short_developed.csv",
    output_dir=r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\stats",
    sep=",",
    col_map={
        "group": "pfas",
        "time": "year",
        "value": "mean_mean"
    },
    year_max=2020,
    min_n=5,
    output_prefix="pfas_mk_pettitt_developed"
)

print(result["trend"])
print(result["change"])

分析完成，结果已保存到: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\stats\pfas_mk_pettitt_developed.xlsx
      group   n    mk_trend   mk_h      mk_p      mk_z    mk_tau  mk_s  \
0  lc_value  18  decreasing   True  0.008015 -2.651439 -0.464052 -71.0   
1  sc_value  18    no trend  False  0.649446  0.454532  0.084967  13.0   
2     value  18    no trend  False  0.324712 -0.984820 -0.176471 -27.0   

   mk_var_s  sen_slope  sen_intercept  
0     697.0  -0.542049      19.362549  
1     697.0   0.076066      20.410796  
2     697.0  -0.429726      39.327655  
      group   n  pettitt_K  pettitt_p  pettitt_cp_index  pettitt_cp_time  \
0  lc_value  18       65.0   0.032556                 4             2007   
1  sc_value  18       42.0   0.358384                 3             2006   
2     value  18       56.0   0.094101                 3             2006   

   seqMK_cp_index  seqMK_cp_time  
0               4           2007  
1               1           2004  
2               3           200

In [4]:

result = run_change_point_analysis(
    file_path=r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_short_developed.csv",
    output_dir=r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\stats",
    sep=",",
    col_map={
        "group": "pfas",
        "time": "year",
        "value": "mean_mean"
    },
    year_max=2020,
    min_n=5,
    output_prefix="pfas_mk_pettitt_developed"
)

print(result["trend"])
print(result["change"])

分析完成，结果已保存到: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\stats\pfas_mk_pettitt_developed.xlsx
      group   n    mk_trend  mk_h      mk_p      mk_z    mk_tau   mk_s  \
0  lc_value  18  decreasing  True  0.000082 -3.939281 -0.686275 -105.0   
1  sc_value  18  decreasing  True  0.001124 -3.257482 -0.568627  -87.0   
2     value  18  decreasing  True  0.000031 -4.166547 -0.725490 -111.0   

   mk_var_s  sen_slope  sen_intercept  
0     697.0  -0.301848      10.660588  
1     697.0  -0.015021       0.717178  
2     697.0  -0.317970      11.400993  
      group   n  pettitt_K  pettitt_p  pettitt_cp_index  pettitt_cp_time  \
0  lc_value  18       80.0   0.003908                 9             2012   
1  sc_value  18       81.0   0.003341                 8             2011   
2     value  18       81.0   0.003341                 8             2011   

   seqMK_cp_index  seqMK_cp_time  
0              10           2013  
1               3           2006  
2              10           201

In [5]:

result = run_change_point_analysis(
    file_path=r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_short_developing.csv",
    output_dir=r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\stats",
    sep=",",
    col_map={
        "group": "pfas",
        "time": "year",
        "value": "mean_mean"
    },
    year_max=2020,
    min_n=5,
    output_prefix="pfas_mk_pettitt_developing"
)

print(result["trend"])
print(result["change"])

分析完成，结果已保存到: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\stats\pfas_mk_pettitt_developing.xlsx
      group   n    mk_trend  mk_h      mk_p      mk_z    mk_tau  mk_s  \
0  lc_value  18  decreasing  True  0.001464 -3.181727 -0.555556 -85.0   
1  sc_value  18  decreasing  True  0.040816 -2.045396 -0.359477 -55.0   
2     value  18  decreasing  True  0.001124 -3.257482 -0.568627 -87.0   

   mk_var_s  sen_slope  sen_intercept  
0     697.0  -0.191630       7.001557  
1     697.0  -0.010100       0.862637  
2     697.0  -0.204898       7.964025  
      group   n  pettitt_K  pettitt_p  pettitt_cp_index  pettitt_cp_time  \
0  lc_value  18       80.0   0.003908                 9             2012   
1  sc_value  18       81.0   0.003341                 8             2011   
2     value  18       80.0   0.003908                 9             2012   

   seqMK_cp_index  seqMK_cp_time  
0               4           2007  
1              15           2018  
2              12           2015  

In [7]:

result = run_change_point_analysis(
    file_path=r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_short_g.csv",
    output_dir=r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\stats",
    sep=",",
    col_map={
        "group": "pfas",
        "time": "year",
        "value": "mean_mean"
    },
    year_max=2020,
    min_n=5,
    output_prefix="pfas_mk_pettitt_g"
)

print(result["trend"])
print(result["change"])

分析完成，结果已保存到: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\stats\pfas_mk_pettitt_g.xlsx
      group   n    mk_trend  mk_h      mk_p      mk_z    mk_tau  mk_s  \
0  lc_value  18  decreasing  True  0.000493 -3.484748 -0.607843 -93.0   
1  sc_value  18  decreasing  True  0.000206 -3.712015 -0.647059 -99.0   
2     value  18  decreasing  True  0.001464 -3.181727 -0.555556 -85.0   

   mk_var_s  sen_slope  sen_intercept  
0     697.0  -0.049570       3.262697  
1     697.0  -0.044454       1.343708  
2     697.0  -0.102478       4.738177  
      group   n  pettitt_K  pettitt_p  pettitt_cp_index  pettitt_cp_time  \
0  lc_value  18       80.0   0.003908                 9             2012   
1  sc_value  18       80.0   0.003908                 9             2012   
2     value  18       80.0   0.003908                 9             2012   

   seqMK_cp_index  seqMK_cp_time  
0              13           2016  
1              13           2016  
2              14           2017  


### 绘图

画不画都无所谓

In [10]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =========================================================
# 1. 读取结果文件
# =========================================================
def load_analysis_result(excel_path):
    """
    读取分析结果 Excel 文件

    Parameters
    ----------
    excel_path : str
        由 run_change_point_analysis 导出的 Excel 路径

    Returns
    -------
    dict
        {
            "filtered_data": DataFrame,
            "trend": DataFrame,
            "change": DataFrame,
            "seqmk": DataFrame
        }
    """
    result = {
        "filtered_data": pd.read_excel(excel_path, sheet_name="filtered_data"),
        "trend": pd.read_excel(excel_path, sheet_name="MK_trend"),
        "change": pd.read_excel(excel_path, sheet_name="ChangePoint"),
        "seqmk": pd.read_excel(excel_path, sheet_name="SeqMK_UF_UB")
    }
    return result


# =========================================================
# 2. 工具函数：确保目录存在
# =========================================================
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


# =========================================================
# 3. 工具函数：安全获取单组结果
# =========================================================
def get_group_change_info(change_df, group_name):
    """
    从 ChangePoint 表中提取指定 group 的突变信息
    """
    sub = change_df[change_df["group"] == group_name]
    if sub.empty:
        return None
    return sub.iloc[0].to_dict()


def get_group_trend_info(trend_df, group_name):
    """
    从 MK_trend 表中提取指定 group 的趋势信息
    """
    sub = trend_df[trend_df["group"] == group_name]
    if sub.empty:
        return None
    return sub.iloc[0].to_dict()


# =========================================================
# 4. 单组时间序列图
# =========================================================
def plot_single_timeseries(
    data_df,
    trend_df=None,
    change_df=None,
    group_name=None,
    output_path=None,
    show_pettitt=True,
    show_seqmk=False,
    show_sen_line=True,
    figsize=(8, 5),
    dpi=150
):
    """
    绘制单个 group 的时间序列图

    Parameters
    ----------
    data_df : DataFrame
        filtered_data，对应列：group, time, value
    trend_df : DataFrame or None
        MK_trend 结果表
    change_df : DataFrame or None
        ChangePoint 结果表
    group_name : str
        分组名称
    output_path : str or None
        输出图片路径；若为 None 则只显示不保存
    show_pettitt : bool
        是否显示 Pettitt 突变时间
    show_seqmk : bool
        是否显示 SeqMK 候选突变时间
    show_sen_line : bool
        是否叠加 Sen slope 趋势线
    """
    g = data_df[data_df["group"] == group_name].sort_values("time").copy()
    if g.empty:
        print(f"[警告] group={group_name} 无数据，跳过绘图")
        return

    x = g["time"].to_numpy()
    y = g["value"].to_numpy()

    plt.figure(figsize=figsize, dpi=dpi)
    plt.plot(x, y, marker="o", linewidth=1.8, label="Value")

    title_lines = [f"Time Series - {group_name}"]

    # 趋势信息
    trend_info = None
    if trend_df is not None:
        trend_info = get_group_trend_info(trend_df, group_name)

    if show_sen_line and trend_info is not None:
        slope = trend_info.get("sen_slope", np.nan)
        intercept = trend_info.get("sen_intercept", np.nan)
        if pd.notna(slope) and pd.notna(intercept):
            y_fit = intercept + slope * x
            plt.plot(x, y_fit, linestyle="--", linewidth=1.5, label="Sen slope line")

    if trend_info is not None:
        mk_trend = trend_info.get("mk_trend", "")
        mk_p = trend_info.get("mk_p", np.nan)
        title_lines.append(f"MK trend: {mk_trend}, p={mk_p:.4f}" if pd.notna(mk_p) else f"MK trend: {mk_trend}")

    # 突变信息
    change_info = None
    if change_df is not None:
        change_info = get_group_change_info(change_df, group_name)

    if change_info is not None:
        if show_pettitt:
            cp_time = change_info.get("pettitt_cp_time", np.nan)
            pettitt_p = change_info.get("pettitt_p", np.nan)
            if pd.notna(cp_time):
                plt.axvline(cp_time, color="red", linestyle="--", linewidth=1.5,
                            label=f"Pettitt CP ({cp_time})")
                title_lines.append(
                    f"Pettitt CP: {cp_time}, p={pettitt_p:.4f}" if pd.notna(pettitt_p)
                    else f"Pettitt CP: {cp_time}"
                )

        if show_seqmk:
            seq_cp_time = change_info.get("seqMK_cp_time", np.nan)
            if pd.notna(seq_cp_time):
                plt.axvline(seq_cp_time, color="purple", linestyle=":", linewidth=1.5,
                            label=f"SeqMK CP ({seq_cp_time})")

    plt.xlabel("Time")
    plt.ylabel("Value")
    plt.title("\n".join(title_lines))
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    if output_path:
        plt.savefig(output_path, bbox_inches="tight")
        plt.close()
    else:
        plt.show()


# =========================================================
# 5. 单组顺序 MK 图
# =========================================================
def plot_single_seqmk(
    seqmk_df,
    change_df=None,
    group_name=None,
    output_path=None,
    show_pettitt=True,
    show_seqmk=True,
    sig_level_line=1.96,
    figsize=(8, 5),
    dpi=150
):
    """
    绘制单个 group 的顺序 MK 图（UF / UB）

    Parameters
    ----------
    seqmk_df : DataFrame
        SeqMK_UF_UB 结果表，对应列：group, time, value, UF, UB
    change_df : DataFrame or None
        ChangePoint 结果表
    group_name : str
        分组名称
    output_path : str or None
        输出图片路径
    show_pettitt : bool
        是否显示 Pettitt 突变时间
    show_seqmk : bool
        是否显示顺序 MK 候选突变时间
    sig_level_line : float
        显著性阈值，默认 1.96
    """
    g = seqmk_df[seqmk_df["group"] == group_name].sort_values("time").copy()
    if g.empty:
        print(f"[警告] group={group_name} 无 SeqMK 数据，跳过绘图")
        return

    x = g["time"].to_numpy()
    uf = g["UF"].to_numpy()
    ub = g["UB"].to_numpy()

    plt.figure(figsize=figsize, dpi=dpi)
    plt.plot(x, uf, marker="o", linewidth=1.6, label="UF")
    plt.plot(x, ub, marker="s", linewidth=1.6, label="UB")

    plt.axhline(0, color="black", linewidth=1)
    plt.axhline(sig_level_line, color="gray", linestyle="--", linewidth=1, label=f"+{sig_level_line}")
    plt.axhline(-sig_level_line, color="gray", linestyle="--", linewidth=1, label=f"-{sig_level_line}")

    title_lines = [f"Sequential Mann-Kendall - {group_name}"]

    change_info = None
    if change_df is not None:
        change_info = get_group_change_info(change_df, group_name)

    if change_info is not None:
        if show_pettitt:
            cp_time = change_info.get("pettitt_cp_time", np.nan)
            pettitt_p = change_info.get("pettitt_p", np.nan)
            if pd.notna(cp_time):
                plt.axvline(cp_time, color="red", linestyle="--", linewidth=1.5,
                            label=f"Pettitt CP ({cp_time})")
                title_lines.append(
                    f"Pettitt CP: {cp_time}, p={pettitt_p:.4f}" if pd.notna(pettitt_p)
                    else f"Pettitt CP: {cp_time}"
                )

        if show_seqmk:
            seq_cp_time = change_info.get("seqMK_cp_time", np.nan)
            if pd.notna(seq_cp_time):
                plt.axvline(seq_cp_time, color="purple", linestyle=":", linewidth=1.5,
                            label=f"SeqMK CP ({seq_cp_time})")
                title_lines.append(f"SeqMK CP: {seq_cp_time}")

    plt.xlabel("Time")
    plt.ylabel("Statistic")
    plt.title("\n".join(title_lines))
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    if output_path:
        plt.savefig(output_path, bbox_inches="tight")
        plt.close()
    else:
        plt.show()


# =========================================================
# 6. 批量绘制时间序列图
# =========================================================
def plot_all_timeseries(
    excel_path,
    output_dir=None,
    show_pettitt=True,
    show_seqmk=False,
    show_sen_line=True,
    figsize=(8, 5),
    dpi=150
):
    """
    从结果 Excel 批量绘制所有 group 的时间序列图
    """
    result = load_analysis_result(excel_path)
    data_df = result["filtered_data"]
    trend_df = result["trend"]
    change_df = result["change"]

    if output_dir is None:
        base_dir = os.path.dirname(excel_path)
        output_dir = os.path.join(base_dir, "timeseries_plots")

    ensure_dir(output_dir)

    groups = sorted(data_df["group"].dropna().unique())

    for group_name in groups:
        safe_name = str(group_name).replace("/", "_").replace("\\", "_").replace(" ", "_")
        out_path = os.path.join(output_dir, f"{safe_name}_timeseries.png")

        plot_single_timeseries(
            data_df=data_df,
            trend_df=trend_df,
            change_df=change_df,
            group_name=group_name,
            output_path=out_path,
            show_pettitt=show_pettitt,
            show_seqmk=show_seqmk,
            show_sen_line=show_sen_line,
            figsize=figsize,
            dpi=dpi
        )

    print(f"时间序列图已输出到: {output_dir}")


# =========================================================
# 7. 批量绘制顺序 MK 图
# =========================================================
def plot_all_seqmk(
    excel_path,
    output_dir=None,
    show_pettitt=True,
    show_seqmk=True,
    sig_level_line=1.96,
    figsize=(8, 5),
    dpi=150
):
    """
    从结果 Excel 批量绘制所有 group 的顺序 MK 图
    """
    result = load_analysis_result(excel_path)
    seqmk_df = result["seqmk"]
    change_df = result["change"]

    if output_dir is None:
        base_dir = os.path.dirname(excel_path)
        output_dir = os.path.join(base_dir, "seqmk_plots")

    ensure_dir(output_dir)

    groups = sorted(seqmk_df["group"].dropna().unique())

    for group_name in groups:
        safe_name = str(group_name).replace("/", "_").replace("\\", "_").replace(" ", "_")
        out_path = os.path.join(output_dir, f"{safe_name}_seqmk.png")

        plot_single_seqmk(
            seqmk_df=seqmk_df,
            change_df=change_df,
            group_name=group_name,
            output_path=out_path,
            show_pettitt=show_pettitt,
            show_seqmk=show_seqmk,
            sig_level_line=sig_level_line,
            figsize=figsize,
            dpi=dpi
        )

    print(f"顺序 MK 图已输出到: {output_dir}")


# =========================================================
# 8. 汇总入口：一键出图
# =========================================================
def plot_all_from_result_excel(
    excel_path,
    output_root=None,
    plot_timeseries=True,
    plot_seqmk=True,
    show_pettitt=True,
    show_seqmk_on_timeseries=False,
    show_seqmk_on_seqfig=True,
    show_sen_line=True,
    sig_level_line=1.96,
    figsize=(8, 5),
    dpi=150
):
    """
    从分析结果 Excel 一键批量绘图

    Parameters
    ----------
    excel_path : str
        分析结果 Excel 路径
    output_root : str or None
        总输出目录；若为 None，则默认在 Excel 同目录下创建 plot_output
    """
    if output_root is None:
        output_root = os.path.join(os.path.dirname(excel_path), "plot_output")

    ensure_dir(output_root)

    if plot_timeseries:
        ts_dir = os.path.join(output_root, "timeseries")
        plot_all_timeseries(
            excel_path=excel_path,
            output_dir=ts_dir,
            show_pettitt=show_pettitt,
            show_seqmk=show_seqmk_on_timeseries,
            show_sen_line=show_sen_line,
            figsize=figsize,
            dpi=dpi
        )

    if plot_seqmk:
        mk_dir = os.path.join(output_root, "seqmk")
        plot_all_seqmk(
            excel_path=excel_path,
            output_dir=mk_dir,
            show_pettitt=show_pettitt,
            show_seqmk=show_seqmk_on_seqfig,
            sig_level_line=sig_level_line,
            figsize=figsize,
            dpi=dpi
        )

    print(f"全部绘图完成，输出目录: {output_root}")

In [12]:

plot_all_from_result_excel(
    excel_path=r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\stats\pfas_mk_pettitt_g.xlsx",
    output_root=r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\stats\plot_output",
    plot_timeseries=True,
    plot_seqmk=True,
    show_pettitt=True,
    show_seqmk_on_timeseries=False,
    show_seqmk_on_seqfig=True,
    show_sen_line=True,
    sig_level_line=1.96
)

时间序列图已输出到: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\stats\plot_output\timeseries
顺序 MK 图已输出到: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\stats\plot_output\seqmk
全部绘图完成，输出目录: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\stats\plot_output
